In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

from langchain_community.vectorstores import FAISS



In [12]:
url1="https://techcrunch.com/2025/12/21/israels-famed-vc-jon-medved-diagnosed-with-als-backed-the-tech-that-will-improve-his-life/"
url2="https://techcrunch.com/2025/12/10/figma-launches-new-ai-powered-object-removal-and-image-extension/"

In [13]:
loader=WebBaseLoader([url1,url2])
data=loader.load()

In [14]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)
chunks=text_splitter.split_documents(data)

In [15]:
embeddings=OpenAIEmbeddings(model="text-embedding-3-large")

In [17]:
import faiss
vector=FAISS.from_documents(chunks,embeddings)
retriever=vector.as_retriever()

In [18]:
prompt_template="""
  answer the question {input} based solely on the context below:
  \n\n<context>\n{context}\n<context>
  if you cant find an answer, say i do not know"""

In [19]:
prompt=PromptTemplate.from_template(prompt_template)

In [20]:
llm=ChatOpenAI(model="gpt-3.5-turbo",temperature=0.0)

In [21]:
combine_docs_chain=create_stuff_documents_chain(llm,prompt)

In [22]:
chain=create_retrieval_chain(retriever,combine_docs_chain)

In [23]:
result=chain.invoke({"input":"what is figma launching"})

In [24]:
result

{'input': 'what is figma launching',
 'context': [Document(id='c4a54397-c67a-464d-9bad-6274ae512f01', metadata={'source': 'https://techcrunch.com/2025/12/10/figma-launches-new-ai-powered-object-removal-and-image-extension/', 'title': 'Figma launches new AI-powered object removal and image extension | TechCrunch', 'description': 'Figma is launching a new image editing toolbar to bring all its features in one place.', 'language': 'en-US'}, page_content='Figma’s launch comes on the same day as Adobe making some of these features available to users within ChatGPT. Figma was one of the launch partners of the app, calling on ChatGPT in October. It is'),
  Document(id='1199acac-975f-431f-bcd3-235899a181bf', metadata={'source': 'https://techcrunch.com/2025/12/10/figma-launches-new-ai-powered-object-removal-and-image-extension/', 'title': 'Figma launches new AI-powered object removal and image extension | TechCrunch', 'description': 'Figma is launching a new image editing toolbar to bring all i